# The Brain Class

A fundamental aspect of deep learning involves iterating through a dataset multiple times and updating model parameters, commonly referred to as the "training loop." To streamline and organize this process, Real-Time-Speech-Separation-Model-Toolkit offers a versatile framework in the form of the "Brain" class, implemented in `real_time_speech_separation_toolkit/core.py`. In each recipe, this class is sub-classed, and its methods are overridden to tailor the implementation to the specific requirements of that recipe.

The core method of the Brain class is `fit()`, responsible for iterating through the dataset, performing updates to the model, and managing the training loop. To leverage `fit()`, at least two methods must be defined in the sub-class: `compute_forward()` and `compute_objectives()`. These methods handle the computation of the model for generating predictions and the calculation of loss terms required for gradient computation.

Let's explore a minimal example to illustrate this:


In [ ]:
%%capture
# Installing Real-Time-Speech-Separation-Model-Toolkit via pip
BRANCH = 'main'
# !python -m pip install git+https://github.com/your-repo/real-time-speech-separation-model-toolkit.git@$BRANCH

# Clone Real-Time-Speech-Separation-Model-Toolkit repository
# !git clone https://github.com/your-repo/real-time-speech-separation-model-toolkit/

In [ ]:
import torch
# import real_time_speech_separation_toolkit as rtsst

# For this example, we'll use a mock Brain class since the actual toolkit isn't installed
class MockBrain:
    def __init__(self, modules, opt_class, hparams=None):
        self.modules = modules
        self.opt_class = opt_class
        self.hparams = hparams or {}
        self.optimizer = None
    
    def fit(self, epoch_counter, data):
        if not self.optimizer:
            self.optimizer = self.opt_class(self.modules['model'].parameters())
        
        for epoch in epoch_counter:
            for batch in data:
                predictions = self.compute_forward(batch, 'train')
                loss = self.compute_objectives(predictions, batch, 'train')
                
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
    
    def compute_forward(self, batch, stage):
        return self.modules["model"](batch["input"])

    def compute_objectives(self, predictions, batch, stage):
        return torch.nn.functional.l1_loss(predictions, batch["target"])

model = torch.nn.Linear(in_features=10, out_features=10)
brain = MockBrain({"model": model}, opt_class=lambda x: torch.optim.SGD(x, 0.1))
data = [{"input": torch.rand(10, 10), "target": torch.rand(10, 10)}]
brain.fit(range(10), data)

With just around 10 lines of code, we can successfully train a neural model. This efficiency is achieved because the Brain class handles intricate details of training, such as managing `train()` and `eval()` states or computing and applying gradients. Furthermore, the flexibility of the class allows every step of the process to be overridden by adding methods to the sub-class. This means that even intricate training procedures, such as those involved in Generative Adversarial Networks (GAN), can be seamlessly integrated into the Brain class.

In this tutorial, we'll begin by elucidating the parameters of the Brain class. Subsequently, we'll delve into the `fit()` method, breaking it down step by step and highlighting the segments that can be overridden when necessary. These insights into the class's parameters and the `fit()` method form the foundation for understanding the functionality and versatility of the Brain class.


## Arguments to `Brain` class

The Brain class only takes 5 arguments, but each of these can be a little complex, so we explain them in detail here. The relevant code is just the `__init__` definition:

```python
def __init__(
    self,
    modules=None,
    opt_class=None,
    hparams=None,
    run_opts=None,
    checkpointer=None,
):
```

### `modules` argument

This first argument takes a dictionary of torch modules. The Brain class takes this dictionary and converts it to a Torch ModuleDict. This provides a convenient way to move all parameters to the correct device, call `train()` and `eval()`, and wrap the modules in the appropriate distributed wrapper if necessary.

### `opt_class` argument

The Brain class takes a function definition for a pytorch optimizer. The reason for choosing this as input rather than a pre-constructed pytorch optimizer is that the Brain class automatically handles wrapping the module parameters in distributed wrappers if requested. This needs to happen before the parameters get passed to the optimizer constructor.

To pass a pytorch optimizer constructor, a lambda can be used, as in the example at the beginning of this tutorial. More convenient, however, is the option used by most of the recipes in Real-Time-Speech-Separation-Model-Toolkit: define the constructor with HyperPyYAML. The `!name:` tag acts similarly to the lambda, creating a new constructor that can be used to make optimizers.

```yaml
optimizer: !name:torch.optim.Adam
    lr: 0.1
```

Of course sometimes zero or multiple optimizers are required. In the case of multiple optimizers, the `init_optimizers` method can be overridden to initialize each individually.


### `hparams` argument

The Brain class algorithm may depend on a set of hyperparameters that should be easy to control externally, this argument accepts a dictionary that will be accessible to all the internal methods using "dot notation". An example follows:

In [ ]:
class SimpleBrain(MockBrain):
  def compute_objectives(self, predictions, batch, stage):
    term1 = torch.nn.functional.l1_loss(predictions, batch["target1"])
    term2 = torch.nn.functional.mse_loss(predictions, batch["target2"])
    return self.hparams.weight1 * term1 + self.hparams.weight2 * term2

hparams = {"weight1": 0.7, "weight2": 0.3}
model = torch.nn.Linear(in_features=10, out_features=10)
brain = SimpleBrain(
  modules={"model": model},
  opt_class=lambda x: torch.optim.SGD(x, 0.1),
  hparams=hparams,
)
data = [{
  "input": torch.rand(10, 10),
  "target1": torch.rand(10, 10),
  "target2": torch.rand(10, 10),
}]
brain.fit(range(10), data)

### `run_opts` argument

There are a large number of options for controlling the execution details for the `fit()` method, that can all be passed via this argument. Some examples include enabling debug mode, the execution device, and the distributed execution options.


### `checkpointer` argument

Finally, if you pass a Real-Time-Speech-Separation-Model-Toolkit checkpointer to the Brain class, there are several operations that automatically get called:

  1. The optimizer parameters are added to the checkpointer.
  2. At the beginning of training, the most recent checkpoint is loaded and training is resumed from that point. If training is finished, this simply ends the training step and moves on to evaluation.
  3. During training, checkpoints are saved every 15 minutes by default (this can be changed or disabled with an option in `run_opts`).
  4. At the beginning of evaluation, the "best" checkpoint is loaded, as determined by the lowest or highest score on a metric recorded in the checkpoints.

## The `fit()` method

This method does a lot, but only actually takes about ~100 lines of code, so it is understandable by reading the code itself. We break it down section-by-section and explain what each part is doing. First, let's briefly go over the arguments:

```python
def fit(
    self,
    epoch_counter,
    train_set,
    valid_set=None,
    progressbar=None,
    train_loader_kwargs={},
    valid_loader_kwargs={},
):
```

1.   The `epoch_counter` argument takes an iterator, so when `fit()` is called, the outer loop iterates this variable. This argument was co-designed with an `EpochCounter` class enabling storage of the epoch loop state. With this argument, we can restart experiments from where they left off.
2.   The `train_set` and `valid_set` arguments take a Torch Dataset or DataLoader that will load the tensors needed for training. If a DataLoader is not passed, one will be constructed automatically (see next section).
3.   The `progressbar` argument controls whether a `tqdm` progressbar is displayed showing progress through the dataset for each epoch.
4.   The `train_loader_kwargs` and `valid_loader_kwargs` are passed to the `make_dataloader` method for making the DataLoader (see next section).



### Fit structure

With the arguments out of the way, we can start to look at the structure of this method. The fit method contains several override-able calls that allow customization of the training process. We'll go over these one-by-one through the rest of the tutorial.

The main components of the fit method are:
1. DataLoader creation
2. Setup operations
3. Epoch loop
4. Validation loop
5. Checkpointing

Each of these components can be customized by overriding specific methods in the Brain subclass.


### `make_dataloader`

The first step of the `fit()` method is to ensure that the data is in an appropriate format for iteration. Both the `train_set` and `valid_set` are passed along with their respective keyword arguments. Here's the actual code structure:

```python
if not isinstance(train_set, DataLoader):
    train_set = self.make_dataloader(
        train_set, stage=rtsst.Stage.TRAIN, **train_loader_kwargs
    )
if valid_set is not None and not isinstance(valid_set, DataLoader):
    valid_set = self.make_dataloader(
        valid_set,
        stage=rtsst.Stage.VALID,
        ckpt_prefix=None,
        **valid_loader_kwargs,
    )
```

By default, this method handles potential complications to DataLoader creation, such as creating a DistributedSampler for distributed execution. As with all of the other methods in the `fit()` call, this can be overridden by creating a `make_dataloader` method in Brain's sub-class definition.

### `on_fit_start`

Besides the dataloader, there's some setup that needs to happen before training can begin. Here's the relevant code:

```python
self.on_fit_start()

if progressbar is None:
    progressbar = self.progressbar
```

The `on_fit_start` method takes care of a few important things, which can most easily be explained by sharing the code structure:

```python
def on_fit_start(self):
    # Handle checkpoint recovery
    if self.checkpointer is not None:
        self.checkpointer.recover_if_possible(
            device=self.device,
            min_keys=None,
        )

    # Initialize optimizers
    self.init_optimizers()

    # Move modules to device
    self.modules.to(self.device)
```

This method handles:
1. Checkpoint recovery (if a checkpointer is provided)
2. Optimizer initialization
3. Moving modules to the correct device

Each of these sub-steps can be overridden as well.

### The Epoch Loop

The main training loop is structured as follows:

```python
for epoch in epoch_counter:
    self.on_epoch_start(epoch)
    
    # Training loop
    self.modules.train()
    for batch in train_set:
        self.on_fit_batch_start(batch)
        
        # Forward pass
        predictions = self.compute_forward(batch, rtsst.Stage.TRAIN)
        
        # Compute loss
        loss = self.compute_objectives(predictions, batch, rtsst.Stage.TRAIN)
        
        # Backward pass
        loss.backward()
        
        # Optimizer step
        self.optimizer.step()
        self.optimizer.zero_grad()
        
        self.on_fit_batch_end(batch, predictions, loss)
    
    # Validation loop
    if valid_set is not None:
        self.evaluate(valid_set)
    
    self.on_epoch_end(epoch)
```

Each of the method calls in this loop can be overridden to customize the training process.

### Customizing the Training Process

The Brain class is designed to be highly customizable. Here are some common customization scenarios:

#### Custom Data Processing
```python
class CustomBrain(rtsst.Brain):
    def on_fit_batch_start(self, batch):
        # Apply custom data augmentation
        batch = self.augment_data(batch)
        return super().on_fit_batch_start(batch)
```

#### Custom Loss Computation
```python
class CustomBrain(rtsst.Brain):
    def compute_objectives(self, predictions, batch, stage):
        # Standard loss
        loss = super().compute_objectives(predictions, batch, stage)
        
        # Add regularization
        if stage == rtsst.Stage.TRAIN:
            reg_loss = self.compute_regularization()
            loss = loss + 0.01 * reg_loss
        
        return loss
```

#### Custom Logging
```python
class CustomBrain(rtsst.Brain):
    def on_fit_batch_end(self, batch, predictions, loss):
        # Log custom metrics
        self.log_metric("custom_metric", self.compute_custom_metric(predictions, batch))
        return super().on_fit_batch_end(batch, predictions, loss)
```

## Advanced Features

### Multiple Optimizers

For complex models like GANs, you might need multiple optimizers:

```python
class GANBrain(rtsst.Brain):
    def init_optimizers(self):
        self.optimizer_G = self.opt_class_G(self.modules.generator.parameters())
        self.optimizer_D = self.opt_class_D(self.modules.discriminator.parameters())
    
    def compute_forward(self, batch, stage):
        # Custom forward pass for GAN
        pass
    
    def fit_batch(self, batch):
        # Custom training step for GAN
        pass
```

### Gradient Clipping

You can easily add gradient clipping:

```python
class ClippedBrain(rtsst.Brain):
    def fit_batch(self, batch):
        predictions = self.compute_forward(batch, rtsst.Stage.TRAIN)
        loss = self.compute_objectives(predictions, batch, rtsst.Stage.TRAIN)
        
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(self.modules.parameters(), max_norm=1.0)
        
        self.optimizer.step()
        self.optimizer.zero_grad()
        
        return loss.detach()
```

## Summary

The Brain class in Real-Time-Speech-Separation-Model-Toolkit provides a powerful and flexible framework for training neural networks. Its key strengths are:

1. **Simplicity**: Basic training can be done with just a few lines of code
2. **Flexibility**: Every aspect of the training process can be customized
3. **Consistency**: Provides a standard interface across all recipes
4. **Extensibility**: Easy to add new features and training strategies

By understanding the structure and override points of the Brain class, you can create sophisticated training pipelines while maintaining clean and readable code.

For more examples and advanced usage patterns, refer to the other tutorials and the recipe implementations in the Real-Time-Speech-Separation-Model-Toolkit repository.